# Functions Example

This notebook demonstrates how to use function calling with GigaChat API.


Import required libraries for function calling and GigaChat LLM.

In [1]:
import os
import getpass
import json
import httpx

from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole, Function, FunctionParameters

Set up GigaChat credentials and scope. The input will be hidden for security.

In [2]:
if "GIGACHAT_CREDENTIALS" not in os.environ:
    os.environ["GIGACHAT_CREDENTIALS"] = getpass.getpass("GigaChat Credentials: ")

if "GIGACHAT_SCOPE" not in os.environ:
    os.environ["GIGACHAT_SCOPE"] = getpass.getpass("GigaChat Scope: ")

GigaChat Credentials:  ········
GigaChat Scope:  ········


Define a function that will be called by the model to get the weather.

In [3]:
def get_weather(location: str, unit: str = "celsius") -> str:
    """Get current weather for a location using wttr.in API."""
    try:
        # Use wttr.in API for weather data (no API key required)
        url = f"https://wttr.in/{location}?format=j1"
        with httpx.Client(timeout=5.0) as client:
            response = client.get(url)
            response.raise_for_status()
            data = response.json()
        
        current = data["current_condition"][0]
        temp_c = current["temp_C"]
        temp_f = current["temp_F"]
        condition = current["weatherDesc"][0]["value"]
        humidity = current["humidity"]
        wind_speed = current["windspeedKmph"]
        
        if unit == "fahrenheit":
            temp = f"{temp_f}°F"
        else:
            temp = f"{temp_c}°C"
        
        result = f"Weather in {location}: {temp}, {condition}, humidity {humidity}%, wind {wind_speed} km/h"
        return result
    except Exception as e:
        return f"Failed to get weather for {location}: {str(e)}"


weather_function = Function(
    name="get_weather",
    description="Get current weather for a location",
    parameters=FunctionParameters(
        type="object",
        properties={
            "location": {
                "type": "string",
                "description": "City name, e.g., New York"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit"
            }
        },
        required=["location"],
    ),
)

**Function Calling Loop**

Set up the function definition and create an interactive loop for function calling.

In [ ]:
with GigaChat(verify_ssl_certs=False) as giga:
    messages = []
    function_called = False
    while True:
        # If the previous LLM response was not a function call - ask the user to continue the dialogue
        if not function_called:
            query = input("\033[92mUser: \033[0m")
            messages.append(Messages(role=MessagesRole.USER, content=query))

        chat = Chat(messages=messages, functions=[weather_function])

        resp = giga.chat(chat).choices[0]
        mess = resp.message
        messages.append(mess)

        print("\033[93m" + f"Bot: \033[0m{mess.content}")

        function_called = False
        func_result = ""
        if resp.finish_reason == "function_call":
            print("\033[90m" + f"  >> Processing function call {mess.function_call}" + "\033[0m")
            if mess.function_call.name == "get_weather":
                location = mess.function_call.arguments.get("location", None)
                unit = mess.function_call.arguments.get("unit", "celsius")
                if location:
                    func_result = get_weather(location, unit)
            print("\033[90m" + f"  << Function result: {func_result}\n\n" + "\033[0m")

            messages.append(
                Messages(role=MessagesRole.FUNCTION, content=json.dumps({"result": func_result}, ensure_ascii=False))
            )
            function_called = True

User:  what is the weather in New York


Bot: 
  >> Processing function call name='get_weather' arguments={'location': 'New York'}
  << Function result: Weather in New York: 5°C, Overcast, humidity 70%, wind 20 km/h


Bot: В Нью-Йорке сейчас пасмурно, температура воздуха составляет около 5°C, влажность воздуха — 70%, дует умеренный ветер со скоростью примерно 20 км/ч.
